# 부두 선석별 월 대기시간 및 재항시간

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding = 'euc-kr')

In [4]:
port1, port2, port3, port4, port5 = [
    load_dataset(f'port{i}') for i in range(1, 6)
]

record = load_dataset("record")

In [5]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

def read_csv_file(filename):
    path = DATA_DIR / filename

    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949")


# port1~5 합치기
port_list = []

for i in range(1, 6):
    p = read_csv_file(f"port{i}.csv")

    # port3만 '선박명'이라는 열 이름을 사용
    p = p.rename(columns={
        "선박명": "선명",
        "선석": "접안자료선석"
    })

    p["부두"] = i
    port_list.append(p)

port = pd.concat(port_list, ignore_index=True)
port = port.drop_duplicates().copy()
port["접안ID"] = range(len(port))

rec = read_csv_file("record.csv").drop_duplicates().copy()


# 날짜 변환
port["접안시각"] = pd.to_datetime(
    port["접안(예정)일시"], format="mixed", errors="coerce"
)
port["선석종료시각"] = pd.to_datetime(
    port["출항(예정)일시"], format="mixed", errors="coerce"
)

rec["입항시각"] = pd.to_datetime(
    rec["입항일시"], format="mixed", errors="coerce"
)
rec["출항시각"] = pd.to_datetime(
    rec["출항일시"], format="mixed", errors="coerce"
)


# record의 계선장소에서 부두·선석 추출
rec[["부두", "선석"]] = (
    rec["계선장소"]
    .str.extract(r"신항\s*([1-5])\s*부두\s*(\d+)\s*선석")
    .apply(pd.to_numeric)
)


# 선명 공백·대소문자 통일
for df in [port, rec]:
    df["연결선명"] = (
        df["선명"].str.upper()
        .str.replace(r"\s+", "", regex=True)
    )


# 필요한 정보가 없는 기록 제외
port = port.dropna(
    subset=["연결선명", "접안시각", "선석종료시각"]
).copy()

rec = rec.dropna(
    subset=["부두", "선석", "연결선명", "입항시각", "출항시각"]
).copy()

port = port[
    port["선석종료시각"] >= port["접안시각"]
].copy()

rec = rec[
    (rec["출항시각"] >= rec["입항시각"])
    & (rec["입항시각"] >= "2022-01-01")
    & (rec["입항시각"] < "2026-01-01")
].copy()

rec[["부두", "선석"]] = rec[["부두", "선석"]].astype(int)

rec["재항시간"] = (
    rec["출항시각"] - rec["입항시각"]
).dt.total_seconds() / 3600

rec["월"] = rec["입항시각"].dt.to_period("M").astype(str)

In [6]:
ships = pd.merge_asof(
    rec.sort_values("출항시각"),

    port[[
        "부두", "연결선명", "접안자료선석",
        "접안시각", "선석종료시각", "접안ID"
    ]].sort_values("선석종료시각"),

    left_on="출항시각",
    right_on="선석종료시각",
    by=["부두", "연결선명"],
    direction="nearest",
    tolerance=pd.Timedelta(hours=24)
)

ships["출항시각차이"] = (
    ships["출항시각"] - ships["선석종료시각"]
).abs()


# 입항 이후, 출항 이전에 접안한 기록만 사용
valid = ships[
    (ships["접안시각"] >= ships["입항시각"])
    & (ships["접안시각"] <= ships["출항시각"])
].copy()


# 같은 접안 기록을 여러 입항에 중복 사용하지 않음
valid = (
    valid.sort_values("출항시각차이")
    .drop_duplicates("접안ID")
    .copy()
)

valid["대기시간"] = (
    valid["접안시각"] - valid["입항시각"]
).dt.total_seconds() / 3600

In [7]:
group_cols = ["부두", "선석", "월"]

# record 전체 기준
monthly_all = (
    rec.groupby(group_cols)
    .agg(
        전체입항건수=("재항시간", "size"),
        전체_평균재항시간=("재항시간", "mean")
    )
)

# 접안 기록까지 연결된 표본 기준
monthly_matched = (
    valid.groupby(group_cols)
    .agg(
        대기시간계산건수=("대기시간", "size"),
        평균대기시간=("대기시간", "mean"),
        연결표본_평균재항시간=("재항시간", "mean")
    )
)

monthly = monthly_all.join(monthly_matched)

monthly["대기시간계산건수"] = (
    monthly["대기시간계산건수"].fillna(0).astype(int)
)

monthly["분석포함률(%)"] = (
    monthly["대기시간계산건수"]
    / monthly["전체입항건수"] * 100
)

monthly = monthly.reset_index().sort_values(group_cols)

display(monthly.round(2))

,부두,선석,월,전체입항건수,전체_평균재항시간,대기시간계산건수,평균대기시간,연결표본_평균재항시간,분석포함률(%)
0,1,1,2022-01,26,24.80,23,0.93,24.85,88.46
1,1,1,2022-02,19,29.99,18,0.34,31.05,94.74
2,1,1,2022-03,24,21.32,24,0.28,21.32,100.00
3,1,1,2022-04,25,25.73,23,0.23,24.51,92.00
4,1,1,2022-05,22,27.97,21,0.35,26.34,95.45
...,...,...,...,...,...,...,...,...,...
908,5,4,2025-08,20,21.66,19,0.39,21.76,95.00
909,5,4,2025-09,23,23.06,23,3.79,23.06,100.00
910,5,4,2025-10,28,23.22,28,3.04,23.22,100.00
911,5,4,2025-11,23,23.78,23,2.29,23.78,100.00


In [9]:
# 1부두 1선석의 월별 결과
display(
    monthly[
        (monthly["부두"] == 1)
        & (monthly["선석"] == 1)
    ].round(2)
)

,부두,선석,월,전체입항건수,전체_평균재항시간,대기시간계산건수,평균대기시간,연결표본_평균재항시간,분석포함률(%)
0,1,1,2022-01,26,24.80,23,0.93,24.85,88.46
1,1,1,2022-02,19,29.99,18,0.34,31.05,94.74
2,1,1,2022-03,24,21.32,24,0.28,21.32,100.00
3,1,1,2022-04,25,25.73,23,0.23,24.51,92.00
4,1,1,2022-05,22,27.97,21,0.35,26.34,95.45
5,1,1,2022-06,20,27.55,19,2.56,27.91,95.00
6,1,1,2022-07,21,28.39,20,1.04,28.36,95.24
7,1,1,2022-08,21,27.86,21,0.30,27.86,100.00
8,1,1,2022-09,19,28.92,19,0.22,28.92,100.00
9,1,1,2022-10,18,25.95,17,0.36,25.09,94.44
